In [ ]:
import model
import data

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
# device="mps"
## Load Dataset
train_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_train.npy", ks_npy="../Data/ks_train.npy")
val_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_val.npy", ks_npy="../Data/ks_val.npy")
test_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_test.npy", ks_npy="../Data/ks_test.npy")

# train_dataset = train_dataset.to(device)
# val_dataset = val_dataset.to(device)
# test_dataset = test_dataset.to(device)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:
temp = next(iter(train_loader))
temp[0].shape, temp[1].shape, temp[2].shape

In [ ]:
m = model.OperatorNetwork(trunk_input_size=7, branch_input_size=6, p = 16)

In [ ]:
criterion = nn.MSELoss()
optimizer = Adam(m.parameters(), lr=0.01)

In [ ]:
import time
import torch
from tqdm import tqdm

EPOCHS = 100
best_val_loss = float('inf')

# Keep track of metrics for plotting later
history = {
    'train_loss': [],
    'val_loss': []
}

print(f"Starting training for {EPOCHS} epochs...")

for e in range(EPOCHS):
    start_time = time.time()
    
    m.train()
    running_train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {e+1}/{EPOCHS} [Train]")
    
    for batch in train_bar:
        branch_input, trunk_input, target = batch

        pred = m(seq=trunk_input, A=branch_input)
        loss = criterion(pred, target)

        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad() 

        batch_size = target.size(0)
        running_train_loss += loss.item() * batch_size
        
        train_bar.set_postfix({'loss': f"{loss.item():.6f}"})
        
    avg_train_loss = running_train_loss / len(train_loader.dataset)
    history['train_loss'].append(avg_train_loss)

    m.eval()
    running_val_loss = 0.0
    
    val_bar = tqdm(val_loader, desc=f"Epoch {e+1}/{EPOCHS} [Val]  ", leave=False)
    
    with torch.no_grad():
        for batch in val_bar:
            branch_input, trunk_input, target = batch
            
            pred = m(seq=trunk_input, A=branch_input)
            val_loss = criterion(pred, target)
            
            batch_size = target.size(0)
            running_val_loss += val_loss.item() * batch_size
            
            val_bar.set_postfix({'val_loss': f"{val_loss.item():.6f}"})
            
    avg_val_loss = running_val_loss / len(val_loader.dataset)
    history['val_loss'].append(avg_val_loss)
    
    epoch_time = time.time() - start_time
    
    print(f"Epoch {e+1:03d}/{EPOCHS} | Time: {epoch_time:.1f}s | "
          f"Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")
          
    if avg_val_loss < best_val_loss:
        improvement = best_val_loss - avg_val_loss
        print(f"   🌟 Val loss improved by {improvement:.6f}! Saving checkpoint...")
        best_val_loss = avg_val_loss
        
        torch.save({
            'epoch': e,
            'model_state_dict': m.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
        }, 'best_deeponet_model.pth')
        